In [7]:
from pathlib import Path
import sys

# Locate 02_Model so that `v3.src` can be imported from anywhere in the repository.
working_directory = Path.cwd().resolve()
model_root = next(
    parent / "02_Model"
    for parent in (working_directory, *working_directory.parents)
    if (parent / "02_Model" / "v3" / "src").is_dir()
)
if str(model_root) not in sys.path:
    sys.path.insert(0, str(model_root))

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from v3.src import create_mines, params
from v3.src.setup_model import build_model
from v3.src.params import Scenario, RefiningRoute

from v3.src import units

world_production_2024_t_li = 222_000.0
manufacturing_efficiency = 0.95  # Matches setup.py.

# Model demand uses final-product proxy tonnes.
final_demand_t = world_production_2024_t_li * units.LCE_PER_LI

config = {
    "initialisation_seed": 2,
    "simulation_seed": 1,
    "end_year": 2050,
    "operating_capacity_multiplier": 1.3,
    "demand_scenario" : "STEPS",
    "refinery_capacity_t_per_year": (
        final_demand_t / manufacturing_efficiency
    ),
}

scenario = Scenario(
    refined_price_per_t=25_000.0,
    mine_markup_fraction=0.0,
    refinery_required_markup_fraction=0.0,
    hard_rock_route=RefiningRoute(
        lithium_recovery_fraction=0.90,
        processing_cost_per_t_output=3_000.0,
    ),
    brine_route=RefiningRoute(
        lithium_recovery_fraction=0.85,
        processing_cost_per_t_output=3_500.0,
    ),
)

In [8]:
raw_data = create_mines.load_raw_mines()

prepared_mines = create_mines.create_mines_dataset(
    raw_data,
    seed=config["initialisation_seed"],
    base_year=params.BASE_YEAR,
    historical_utilisation=params.HISTORICAL_UTILISATION,
)

from v3.src.iea_demand import load_iea_demand

iea = load_iea_demand()

demand_t_li = (
    iea.loc[iea["scenario"].eq(config["demand_scenario"])]
    .set_index("year")["demand_t_li"]
    .copy()
)

# Observed-production benchmark for the model's starting year.
demand_t_li.loc[2024] = world_production_2024_t_li

years = range(params.BASE_YEAR, config["end_year"] + 1)
demand_t_li = demand_t_li.reindex(years)

if demand_t_li.isna().any():
    raise ValueError("Demand is unavailable for some requested years.")

# Convert contained lithium to the existing final-product proxy units.
demand = {
    int(year - params.BASE_YEAR): float(quantity * units.LCE_PER_LI)
    for year, quantity in demand_t_li.items()
}
# Always create fresh assets when changing the configuration.
model = build_model(
    prepared_mines,
    demand=demand,
    refinery_capacity=max(demand.values()) / manufacturing_efficiency,
    simulation_seed=config["simulation_seed"],
    scenario=scenario, 
    start_year=params.BASE_YEAR
)
for mine in model.mines:
    if mine.status == "operation":
        mine.capacity *= config["operating_capacity_multiplier"]

results = model.run()

trades = pd.DataFrame(
    model.transactions,
    columns=pd.Index([
        "period",
        "material_form",
        "source_asset_id",
        "quantity_t_input",
        "output_t",
        "price_per_t_input",
        "transaction_value",
    ]),
)
# Upstream purchases have an associated refined-output equivalent.
upstream = trades.loc[trades["output_t"].notna()].copy()

results

,period,year,opened_mine_ids,demand_t,refined_output_t,manufacturing_input_purchased_t,cathode_output_t,delivered_t,unmet_demand_t,in_use_t,price_usd_per_t_lce,committed_mine_ids,mines_operating
0,0,2024,[],1181706.0,1.243901e+06,1.243901e+06,1.181706e+06,1.181706e+06,0.000000e+00,1.181706e+06,9389.136476,"[13, 19, 20, 21, 27, 29, 32092, 37194, 37597, ...",47
1,1,2025,"[77553, 92723]",1463825.0,1.540868e+06,1.540868e+06,1.463825e+06,1.463825e+06,0.000000e+00,2.645531e+06,12441.088077,"[22, 25, 26, 30, 35788, 37316, 37607, 37630, 3...",49
2,2,2026,"[15, 36153, 79246, 88897, 91034]",1677809.6,1.734546e+06,1.734546e+06,1.647819e+06,1.647819e+06,2.999098e+04,4.293350e+06,14051.212490,"[17, 31477, 32093, 36265, 37315, 37470, 37471,...",54
3,3,2027,"[36750, 78022, 78031, 78109, 78645, 79424, 88960]",1891794.2,1.761896e+06,1.761896e+06,1.673801e+06,1.673801e+06,2.179930e+05,5.967151e+06,15395.155583,"[16, 23, 24, 34639, 36132, 36162, 37468, 37599...",61
4,4,2028,"[37628, 77514, 91065]",2105778.8,1.802683e+06,1.802683e+06,1.712549e+06,1.712549e+06,3.932301e+05,7.679700e+06,16382.281237,"[18, 29806, 36506, 37452, 37518, 37701, 41758,...",64
5,5,2029,[80026],2319763.4,1.717393e+06,1.717393e+06,1.631523e+06,1.631523e+06,6.882401e+05,9.311223e+06,17900.038428,"[37507, 37516, 37797, 38320, 41752, 66225, 663...",65
6,6,2030,[],2533748.0,1.710652e+06,1.710652e+06,1.625120e+06,1.625120e+06,9.086282e+05,1.093634e+07,18754.877533,"[39374, 78420, 79948, 80040, 87450, 87946, 896...",65
7,7,2031,"[77631, 78769, 81348, 81759, 89801, 91264, 914...",2790316.6,1.766938e+06,1.766938e+06,1.678591e+06,1.678591e+06,1.111726e+06,1.261493e+07,19304.466331,"[59387, 59623, 78112, 78206, 81540, 88127, 889...",74
8,8,2032,"[21, 37597, 37607, 37627, 60345, 70738, 77062,...",3046885.2,1.879466e+06,1.879466e+06,1.785492e+06,1.785492e+06,1.261393e+06,1.440043e+07,18862.718441,"[27945, 59768, 80115, 81872, 91720, 91750, 92541]",100
9,9,2033,"[13, 20, 22, 26, 29, 37639, 38700, 40251, 4178...",3303453.8,2.272937e+06,2.272937e+06,2.159290e+06,2.159290e+06,1.144164e+06,1.655972e+07,16750.107188,"[37725, 77600, 79205, 91224, 91321, 91757, 92818]",148


In [9]:
route_labels = {
    "spodumene_concentrate": "Hard-rock concentrate",
    "concentrated_brine": "Concentrated brine",
}
route_colours = {
    "spodumene_concentrate": "#2166ac",
    "concentrated_brine": "#e69f00",
}

periods = results["period"]

tonnes_proxy_per_kt_li = 1000.0 * units.LCE_PER_LI

year_by_period = results.set_index("period")["year"]

# Accepted sellers receive a common price within each material.
# Leave periods without trade as missing prices.
prices = (
    upstream.groupby(["period", "material_form"])["price_per_t_input"]
    .first()
    .unstack("material_form")
    .reindex(index=results["period"], columns=list(route_labels))
    .set_axis(results["year"], axis=0)
)

# Express both routes on the same refined-output basis.
volumes = (
    upstream.groupby(["period", "material_form"])["output_t"]
    .sum()
    .unstack("material_form")
    .reindex(index=results["period"], columns=list(route_labels))
    .fillna(0.0)
    .set_axis(results["year"], axis=0)
)

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Final demand and deliveries",
        "Upstream transaction prices",
        "Purchased supply by route",
        "Accumulated end-use stock",
    ),
    vertical_spacing=0.18,
)

for column, label, colour in [
    ("demand_t", "Demand", "#555555"),
    ("delivered_t", "Delivered", "#228833"),
]:
    fig.add_trace(
        go.Scatter(
            x=results["year"],
            y=results[column] / tonnes_proxy_per_kt_li,
            name=label,
            mode="lines+markers",
            line={"color": colour},
        ),
        row=1, col=1,
    )

for material, label in route_labels.items():
    fig.add_trace(
        go.Scatter(
            x=results["year"],
            y=prices[material],
            name=label,
            legendgroup=material,
            mode="lines+markers",
            connectgaps=False,
            line={"color": route_colours[material]},
        ),
        row=1, col=2,
    )

    fig.add_trace(
        go.Bar(
            x=results["year"],
            y=volumes[material] / tonnes_proxy_per_kt_li,
            name=label,
            legendgroup=material,
            showlegend=False,
            marker_color=route_colours[material],
        ),
        row=2, col=1,
    )

fig.add_trace(
    go.Scatter(
        x=results["year"],
        y=results["in_use_t"] / tonnes_proxy_per_kt_li,
        name="In use",
        mode="lines+markers",
        line={"color": "#884488"},
    ),
    row=2, col=2,
)

fig.update_yaxes(
    title_text="kt Li", row=1, col=1
)
fig.update_yaxes(
    title_text="USD/t physical input", row=1, col=2
)
fig.update_yaxes(
    title_text="kt Li, refined-output equivalent", row=2, col=1
)
fig.update_yaxes(
    title_text="kt Li", row=2, col=2
)
fig.update_xaxes(title_text="Year", dtick=5)
fig.update_yaxes(rangemode="tozero")

fig.update_layout(
    title="Supply-chain experiment",
    template="plotly_white",
    height=750,
    barmode="stack",
    hovermode="x unified",
)

fig.show()